In [ ]:

import pandas as pd

file_path = "dukes_raw/DUKES_5.10_Plant loads, demand and efficiency.xlsx"

excel_file = pd.ExcelFile(file_path)

for idx, sheet_name in enumerate(excel_file.sheet_names, 1):
    print(f"{idx}. {sheet_name}")

In [ ]:
import pandas as pd
import os

FILE = "dukes_raw/DUKES_5.10_Plant loads, demand and efficiency.xlsx"
SHEET = "5.10.B and 5.10.C"

df_full = pd.read_excel(FILE, sheet_name=SHEET, header=None)

header_row = 6
data_start = 7
data_end = 13

header = df_full.iloc[header_row].tolist()

df = df_full.iloc[data_start:data_end + 1].copy()
df.columns = header

df = df.rename(columns={df.columns[0]: "source_tech"})

source_to_tech = {
    "Combined cycle gas turbine stations": "gas",
    "Nuclear stations": "nuclear",
    "Pumped storage hydro": "storage",
    "Conventional thermal and other stations [note 5][note 6]": "other_thermal",
    "of which coal-fired stations [note 7]": "coal",
}

df["source_tech"] = df["source_tech"].astype(str).str.strip()
df = df[df["source_tech"].isin(source_to_tech.keys())].copy()
df["tech"] = df["source_tech"].map(source_to_tech)

rename_year_cols = {}
year_cols = []

for col in df.columns:
    if col not in ["source_tech", "tech"]:
        clean_year = str(col).split("[")[0].replace(".0", "").strip()
        if clean_year.isdigit():
            y = int(clean_year)
            rename_year_cols[col] = y
            if 2010 <= y <= 2024:
                year_cols.append(y)

df = df.rename(columns=rename_year_cols)

df_long = df.melt(
    id_vars=["source_tech", "tech"],
    value_vars=year_cols,
    var_name="year",
    value_name="load_factor_pct"
)

df_long["year"] = df_long["year"].astype(int)
df_long["load_factor_pct"] = pd.to_numeric(df_long["load_factor_pct"], errors="coerce")

df_final = (
    df_long
    .dropna(subset=["load_factor_pct", "tech"])
    .groupby(["year", "tech"], as_index=False)
    .agg(
        load_factor_pct=("load_factor_pct", "mean"),
        source_table=("source_tech", lambda s: "DUKES 5.10.B: " + "; ".join(sorted(set(s))))
    )
)

df_final = df_final[["year", "tech", "load_factor_pct", "source_table"]]
df_final = df_final.sort_values(["year", "tech"]).reset_index(drop=True)

assert df_final["year"].min() == 2010
assert df_final["year"].max() == 2024
assert not df_final.duplicated(["year", "tech"]).any()

os.makedirs("output", exist_ok=True)

df_final.to_parquet("output/dukes_loadfactor_hist_2010_2024.parquet", index=False)

print("5.10B complete")
print("Rows:", len(df_final))
print("Duplicate year+tech rows:", df_final.duplicated(["year", "tech"]).sum())
print("Columns:", df_final.columns.tolist())
print("Years:", df_final.year.min(), "-", df_final.year.max())
print("Tech:", sorted(df_final["tech"].unique()))